# Topics, Partitions & Storage

## What's covered

- Topic naming — conventions that age well
- Sizing the partition count — the throughput math and what makes it hard to change later
- Retention policies — time-based (`retention.ms`) and size-based (`retention.bytes`)
- `cleanup.policy` — `delete` vs `compact` vs both
- Log compaction — tombstones, the deduplication invariant, and what it's actually for
- Segments on disk — the `.log` / `.index` / `.timeindex` file trio
- Replication factor, the ISR, `min.insync.replicas` — the broker-side guarantees
- Rack awareness — surviving an availability-zone failure
- Tiered storage — what changed in Kafka 3.6+ and when to consider it
- The admin surface — create / alter / describe / delete topics, `DeleteRecords`, `ListOffsets`
- Common gotchas

## Topic naming

Topic names live forever once consumers depend on them, so spend a minute on the convention before you create the first one. Three things to nail down up front:

- **Use a delimiter consistently** — `.` or `_`. The dot is the most common (`payments.created.v1`), and Kafka itself uses `__` (double underscore) as a reserved prefix for internal topics like `__consumer_offsets`. Avoid mixing dots and underscores in the same name; Kafka metrics munge both into the same form, which causes monitoring collisions.
- **Encode the producer, entity, and event** — something like `<domain>.<entity>.<event>`. `payments.transaction.created`, `orders.order.cancelled`. This survives reorganization better than naming by team or system.
- **Version in the name, not the payload** — `payments.transaction.created.v1`. When the schema breaks compatibility, run `v1` and `v2` side by side for the migration window. Versioning *inside* the record forces every consumer to handle every version forever.

Length cap: 249 characters. Allowed characters: ASCII letters, digits, `.`, `_`, `-`. No spaces, no slashes.

## Sizing the partition count

Partition count is the single hardest topic design choice. Three forces push in opposite directions:

**More partitions = more parallelism.** Each partition can be read by exactly one consumer per group. Six partitions cap your consumer group at six in-parallel readers. Twelve partitions let you scale to twelve. Pick a number that's at least the largest consumer group you ever expect — including future ones.

**More partitions = more overhead.** Each partition is a set of open files on every replica, a queue in the producer's accumulator, a tracked offset in `__consumer_offsets`, and a metadata entry on the controller. A cluster with one million partitions is a real number you can scale to with KRaft, but it's not free — leader elections take longer, request batching gets worse, controller failover slows down.

**Partitions can be added, but not without breaking things.** Adding partitions to an existing topic is a one-line admin call — but the key-to-partition hash changes for some keys, so per-key ordering is broken for those keys going forward. **You cannot reduce partition count.** A topic with too many partitions is permanent until you create a new topic and migrate.

**The rule of thumb:** size for peak throughput plus headroom, then *don't add more.* A common starting heuristic:

```text
  partition_count = max(
    target_throughput_MBps / per_partition_throughput_MBps,
    peak_consumer_count
  )
```

Per-partition throughput on commodity hardware is roughly 10-50 MB/s sustained (varies wildly with record size, compression, and disk). For most application topics, 12–24 partitions is enough; for firehose topics (clickstream, IoT), the count climbs. Spend more time getting this right at topic-creation than fixing it later.

## Setup

Same broker as before. We'll create a few new topics with explicit configs to demonstrate retention, compaction, and the admin API.

In [ ]:
from confluent_kafka import Producer, Consumer, TopicPartition, OFFSET_BEGINNING
from confluent_kafka.admin import (
    AdminClient, NewTopic, ConfigResource, ConfigEntry, ResourceType,
)

BOOTSTRAP = "localhost:9092"
admin = AdminClient({"bootstrap.servers": BOOTSTRAP})

def ensure_topic(name, partitions=3, rf=1, config=None):
    nt = NewTopic(name, num_partitions=partitions, replication_factor=rf, config=config or {})
    futures = admin.create_topics([nt])
    for n, f in futures.items():
        try: f.result(); print(f"created  {n}")
        except Exception as e:
            if "already exists" in str(e) or "TopicExistsError" in str(type(e)):
                print(f"exists   {n}")
            else:
                raise

## Retention — time and size

Kafka keeps records around until a retention policy removes them. Two knobs, evaluated independently per partition; whichever fires first wins:

| Config | Default | Meaning |
|---|---|---|
| `retention.ms` | `604800000` (7 days) | Maximum age of a record before it's eligible for deletion |
| `retention.bytes` | `-1` (unlimited) | Maximum total bytes per partition before old segments are deleted |

Retention is enforced at the **segment** level, not the record level. Kafka deletes whole segment files when their newest record crosses the threshold — so you'll see records persist a bit past the strict deadline (until the segment they live in fully ages out). The segment roll knobs `segment.ms` and `segment.bytes` indirectly control how tight the deletion granularity is.

Setting `retention.ms=-1` disables time retention entirely — records stay forever (or until size retention kicks in). Useful for compacted topics (next section) where the retention story is replaced by deduplication.

Below: a topic with a short five-minute retention. You'd never use this in production; it's just visible enough to confirm the config landed.

In [ ]:
ensure_topic("events-short-retention", partitions=3, config={
    "retention.ms":    str(5 * 60 * 1000),   # 5 minutes
    "retention.bytes": "104857600",          # 100 MB per partition cap
    "segment.ms":      str(60 * 1000),       # roll segments every minute (so retention is granular)
})

## `cleanup.policy` — delete vs compact

Two cleanup policies, and the choice between them is *the* shaping decision for what a topic means:

- **`cleanup.policy=delete`** (default). Records are deleted by retention. The topic is an **event log** — every record is a discrete happening, history matters, replay reconstructs the past.
- **`cleanup.policy=compact`**. Records are deduplicated by key — only the *latest* value for each key is guaranteed to survive. The topic is a **changelog** — current state per key, with old versions garbage-collected.
- **`cleanup.policy=compact,delete`**. Both. Compaction keeps the latest value per key *within* the retention window. Useful when you want current-state semantics but also a time-bounded history.

The decision is about meaning, not performance:

| Question | Use `delete` | Use `compact` |
|---|---|---|
| Each record is a fact, history matters | ✓ | |
| Only latest state per key matters | | ✓ |
| Want to replay a new consumer from offset 0 to reconstruct history | ✓ | |
| Want a new consumer to read once and have current state | | ✓ |
| Examples | `payments.created`, `clicks` | `users.profile`, `inventory.level`, `__consumer_offsets` |

Kafka itself uses compacted topics for its own state — `__consumer_offsets` is compacted because only the latest committed offset per `(group, topic, partition)` matters.

## Log compaction — how it actually works

Compaction runs as a background thread on each broker (the **log cleaner**). For each compacted topic, it periodically scans old segments and rewrites them to keep only the *latest* value per key.

```text
  before compaction (one partition, by offset)
  ┌──┬──┬──┬──┬──┬──┬──┬──┐
  │A │B │A │C │B │A │D │B │      values omitted; letters are KEYS
  └──┴──┴──┴──┴──┴──┴──┴──┘
    0  1  2  3  4  5  6  7

  after compaction
  ┌────┬──┬──┬────┬────┬──┬──┬──┐
  │ 2A │  │  │ 3C │    │5A│6D│7B│      offsets are PRESERVED — only contents are dropped
  └────┴──┴──┴────┴────┴──┴──┴──┘
    2     3            5  6  7
```

Two important properties:

- **Offsets are preserved.** A compacted partition is still strictly ordered by offset; gaps just appear where superseded records used to be. Consumers do not see the gaps as errors.
- **The active (tail) segment is never compacted.** Records at the very tip of the log are always kept until they age into a closed segment. So a brand-new value with the same key as an older one will coexist with it until the next compaction pass.

**Tombstones — how you delete.** Producing a record with a non-null key and a **`null` value** is a **tombstone**. Compaction treats it as "delete this key," and after `delete.retention.ms` (default 24 hours) the tombstone itself is removed. That delay exists so consumers reading from older offsets get a chance to see the deletion before it disappears.

Two tuning knobs you'll see:

- **`min.cleanable.dirty.ratio`** (default `0.5`) — fraction of the log that must be "dirty" (eligible-but-not-yet-compacted) before compaction runs. Lower values make compaction more aggressive (and more CPU).
- **`min.compaction.lag.ms`** — minimum age of a record before compaction can eat it. Prevents racing with consumers that haven't caught up.

In [ ]:
# Create a compacted topic and produce updates + a tombstone for the same key.
# Note: compaction is asynchronous and won't necessarily fire in the seconds
# this notebook runs in. We're showing the API surface; on a busy production
# topic the log cleaner thread does its work continuously.
ensure_topic("users-profile", partitions=1, config={
    "cleanup.policy":          "compact",
    "min.cleanable.dirty.ratio": "0.1",   # more aggressive than default
    "segment.ms":              "60000",   # close segments quickly so they become cleanable
    "delete.retention.ms":     "60000",   # short tombstone retention for demo
})

p = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True})

# Three updates for one user, then a tombstone.
for value in ("name=Aarav, city=Mumbai",
              "name=Aarav, city=Pune",
              "name=Aarav, city=Bengaluru"):
    p.produce("users-profile", key="u-1", value=value)

p.produce("users-profile", key="u-1", value=None)   # tombstone — "delete u-1"
p.flush()
print("produced 3 updates and 1 tombstone for key=u-1")

## Segments on disk

Each partition's log is split into a sequence of **segments** — fixed-size files on disk. A typical partition directory looks like:

```text
  /var/kafka-logs/payments-0/
  ├── 00000000000000000000.log         ← record data
  ├── 00000000000000000000.index       ← offset → byte position
  ├── 00000000000000000000.timeindex   ← timestamp → offset
  ├── 00000000000123456789.log
  ├── 00000000000123456789.index
  └── 00000000000123456789.timeindex   ← ("active" segment — being appended to)
```

The filename is the **base offset** of the segment — the offset of the first record inside. Once a segment fills up (`segment.bytes`, default 1 GB) or ages past `segment.ms` (default 7 days), it's closed and a new active segment starts.

Three files per segment:

- **`.log`** — the actual records, in produce order, with their headers, keys, values, timestamps.
- **`.index`** — a sparse map from offset → byte position within the `.log`. Lets a fetch for offset N jump directly to roughly the right byte without scanning.
- **`.timeindex`** — a sparse map from timestamp → offset. Powers `offsets_for_times()` from the consumer side.

Why segments matter operationally: **retention and compaction operate on whole segments**. Old records vanish when their *segment file* is deleted (or rewritten by compaction). If `segment.ms` is a week, then setting `retention.ms` to a day still keeps records up to two days, because the segment holding them won't close until its week is up. Tune segment size and retention together.

## Replication factor, ISR, `min.insync.replicas`

Notebook 01 introduced replication; here are the three knobs you actually set when creating a production topic:

- **`replication.factor`** (per-topic) — number of copies of each partition. **3 is the production standard.** 2 is fragile (one broker death = single point of failure); 1 is for development only.
- **`min.insync.replicas`** (per-topic or broker default) — the minimum number of in-sync replicas required to accept a write when the producer uses `acks=all`. **Setting `min.insync.replicas=2` with `replication.factor=3` is the standard recipe.** That gives you one broker of headroom: if any single broker dies, the remaining two still satisfy the minimum, and writes continue. Lose a second broker, and writes fail (the cluster preserves correctness by refusing to accept writes that wouldn't survive a further failure).
- **`unclean.leader.election.enable`** (broker level, **default `false`** since 2.4) — whether a non-ISR replica can be elected leader if all ISR replicas are down. `true` trades data loss for availability; `false` trades availability for correctness. **Leave it `false`.**

The combination `replication.factor=3` / `min.insync.replicas=2` / `acks=all` / `unclean.leader.election.enable=false` is the standard high-durability configuration. Memorize it.

## Rack awareness — surviving an AZ failure

By default, Kafka may place all three replicas of a partition on three brokers in the same availability zone. If that AZ goes down, the partition is offline.

**Rack awareness** fixes this. Each broker is tagged with a `broker.rack` value (typically the AZ name: `us-east-1a`, `us-east-1b`, `us-east-1c`). When the controller assigns replicas, it tries to spread them across distinct racks. A 3-replica partition on a 3-AZ cluster ends up with one replica per AZ — survive any single AZ failure.

Two notes:

- Cross-AZ traffic costs money in AWS/GCP/Azure. Rack awareness creates more cross-AZ replication and more cross-AZ fetches. **Fetch-from-follower** (`replica.selector.class=RackAwareReplicaSelector` + matching `client.rack` on the consumer) lets consumers read from a same-AZ follower replica instead of always going to the leader, which is the main cost mitigation.
- Managed offerings (MSK, Confluent Cloud) configure rack awareness for you. Self-managed clusters in cloud need to set it explicitly.

## Tiered storage — Kafka as cold storage

Historically, every byte Kafka stored sat on broker-local disks. To keep months of history, you sized the cluster for the *total* retained data, even though only the recent tail was actively read. Most of that disk was idle, and growing the cluster meant adding broker hardware you didn't need for throughput.

**Tiered storage** (introduced in Kafka 3.6, GA in 3.7) splits each partition into two tiers:

- **Local tier** — the most recent records, served from broker-local disk (fast).
- **Remote tier** — older records, offloaded to object storage (S3, GCS, Azure Blob). Cheaper, effectively infinite, slower.

Consumers fetch transparently — the broker pulls from the remote tier on demand. The two knobs you set are `local.retention.ms` (how long to keep on local disk) and `retention.ms` (how long total, local + remote).

What changes for you:

- **Massive retention becomes cheap.** Years of history at S3 prices, not EBS prices.
- **Cluster sizing decouples from retention.** Add brokers for throughput, not storage.
- **Replay from the remote tier is slower.** A consumer reading from year-old data hits the object store; expect tens to hundreds of milliseconds per fetch instead of single-digit.

Tiered storage is *opt-in per topic* — you set `remote.storage.enable=true` on the topics you want offloaded. Most application topics don't need it; firehose topics with long retention windows do.

## The admin surface

`AdminClient` is how you manage topics from code. The operations you'll reach for most:

| Method | What it does |
|---|---|
| `create_topics([NewTopic])` | Create one or more topics with partitions, RF, and per-topic config |
| `delete_topics([name])` | Delete topics (asynchronous on the broker — data removal can lag) |
| `list_topics()` | Snapshot of cluster metadata: topics, partitions, leaders, ISR |
| `describe_configs([ConfigResource])` | Read effective configs for a topic or broker |
| `alter_configs([ConfigResource])` | Change topic or broker configs (deprecated; prefer the incremental variant) |
| `incremental_alter_configs([...])` | Set / append / subtract / delete specific config entries — safer than the full replace |
| `create_partitions(...)` | Increase a topic's partition count (cannot decrease) |
| `delete_records(...)` | Delete records up to a given offset per partition — used for GDPR / right-to-erasure workflows |
| `list_offsets(...)` | Earliest / latest / by-timestamp offsets per partition |

Below: read back the effective config of the topic we just created.

In [ ]:
# Describe the effective configs of the compacted topic we created above.
resource = ConfigResource(ResourceType.TOPIC, "users-profile")
future = admin.describe_configs([resource])[resource]
config = future.result()

interesting = ("cleanup.policy", "retention.ms", "retention.bytes",
               "segment.ms", "segment.bytes",
               "min.cleanable.dirty.ratio", "delete.retention.ms",
               "min.insync.replicas")

print(f"{'config':<32}  {'value':<25}  source")
print("-" * 80)
for name in interesting:
    entry = config.get(name)
    if entry is None: continue
    print(f"{name:<32}  {str(entry.value):<25}  {entry.source}")

In [ ]:
# Snapshot cluster metadata — topic count, partition count, leader/ISR per partition.
md = admin.list_topics(timeout=5).topics

print(f"{'topic':<30}  {'partitions':>10}  {'leaders':>10}")
print("-" * 55)
for name, tmeta in sorted(md.items()):
    if name.startswith("__"): continue   # skip internal topics
    leaders = {p.leader for p in tmeta.partitions.values()}
    print(f"{name:<30}  {len(tmeta.partitions):>10}  {len(leaders):>10}")

## Common gotchas

- **Picking partition count without thinking about consumer scaling.** Add headroom: pick the largest plausible consumer count, then a little more. Reducing partitions later is impossible without a topic migration.
- **`segment.ms` longer than `retention.ms`.** Retention can't take effect inside an open segment. A topic with `retention.ms=1h` and `segment.ms=7d` will hold records for up to a week. Set `segment.ms` ≤ `retention.ms`.
- **Producing a null value to a non-compacted topic.** No special meaning — it's just a record with no payload. Tombstones only matter for compacted topics.
- **Tombstones for keys that never existed.** Compaction will still keep them around for `delete.retention.ms`; consumers will see `(key, None)` records. Filter on the consumer side if the application doesn't care.
- **Setting `unclean.leader.election.enable=true` to "avoid downtime."** You're trading silent data loss for uptime. The exam expects you to know this is wrong by default.
- **Same partition count across topics.** Tempting for consistency, but topics have very different traffic. Size each one independently.
- **Forgetting to create topics explicitly.** Many production clusters disable auto-create. The first `produce()` to a missing topic fails with `UnknownTopicOrPartitionError`. Always create with `AdminClient`.

## What's next

You now have the topic-design checklist: name, partition count, replication factor, `min.insync.replicas`, retention, cleanup policy, and the segment knobs underneath them.

- **Notebook 05 — Schema Registry & Serialization.** Opaque-bytes is a footgun at scale; the next notebook covers Avro, Protobuf, JSON Schema, the Confluent wire format, and schema evolution rules that determine whether a topic upgrade is a non-event or an outage.
- **Notebook 06 — Kafka Connect.** Source connectors that pull data into Kafka, sink connectors that push it out, all running as a fleet of long-lived workers — so your application code can stop being a CDC or warehouse-loader.
- **Notebook 07 — Kafka Streams.** Stateful stream processing over the topics you've now designed properly.